# Building and Deploying a Secure MCP Server on Google Cloud Run

Live demo companion for the **Build With AI 2026** talk, presented at Google Developer Group Kuala Lumpur.

**Speaker:** Gregory Tan, AI Security Engineer @ YTL AI Labs

---

This notebook walks through the full lifecycle of a secure ShopApp MCP server — from local development to a production-hardened deployment on Google Cloud Run.

### What you'll build

A Gemini-powered AI shopping assistant backed by an MCP server, secured with:
- **Authentication** via Google Cloud Run IAM (blocks unauthenticated requests before they reach your server)
- **Authorization** via `tool_filter` (limits each agent to only the tools it needs)
- **Tenant Isolation** via SPIFFE identity on Vertex AI Agent Engine (prevents cross-tenant data access)

### Sections

| # | Section | Description |
|---|---------|-------------|
| 1 | Direct MCP Client | Connect to the MCP server, list tools, and invoke them directly |
| 2 | AI Agent (Google ADK) | Use a Gemini agent to interact with MCP tools conversationally |
| 3 | Authentication | Deploy to Cloud Run and enforce IAM token validation |
| 4 | Authorization | Scope agents to specific tools using `tool_filter` |
| 5 | Tenant Isolation | Enforce per-tenant data isolation using SPIFFE identity via Vertex AI Agent Engine |

### MCP Tools

The server exposes three tools on a SQLite-backed ShopApp:
- `search_products(query)` — search the product catalogue
- `get_user(user_id)` — retrieve a user profile  
- `place_order(user_id, product_id, quantity)` — place a product order

---
### Section 1: Directly Invoking MCP Client

Directly connect to the MCP server, list available tools, and call a tool directly.

---

In [ ]:
# (01) Installing Necessary Libraries
!pip install -r requirements.txt -q

In [ ]:
# (01) Importing Necessary Libraries
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
import os, subprocess, warnings, logging
import nest_asyncio
nest_asyncio.apply()

warnings.filterwarnings("ignore")
logging.getLogger("google.adk").setLevel(logging.ERROR)
logging.getLogger("google.genai").setLevel(logging.ERROR)

In [ ]:
# (02) Start MCP server in the background
MCP_SERVER_FILE = "mcp_server.py"
LOG_FILE = "output.log"

os.system(f"pkill -f {MCP_SERVER_FILE}")
open(LOG_FILE, "w").close()

process = subprocess.Popen(
    ["python", MCP_SERVER_FILE],
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT
)
print(f"MCP Server Started | PID {process.pid}")

In [ ]:
# (02) Check if MCP server is running
!tail -n 20 $LOG_FILE

In [ ]:
# (Optional) Stop MCP server
# process.terminate()

In [ ]:
# (03) Interact with MCP Server
async with streamablehttp_client("http://127.0.0.1:8000/mcp") as (reader, writer, _):
    async with ClientSession(reader, writer) as session:
        await session.initialize()
        print("Connected to MCP server!\n")

        tools_list = await session.list_tools()
        print("Available tools:")
        for tool in tools_list.tools:
            print(f"  - {tool.name}: {tool.description}")

---
### Section 2: Using AI Agents (Google ADK) to Chat with MCP

This section demonstrates how to use an AI agent (via Google ADK) to interact with MCP tools in a conversational way.

---

Before running the cells below, fill in your GCP project details. If you are unsure where to find your **Project ID**, refer to the [GCP Prerequisites guide](https://docs.cloud.google.com/vertex-ai/docs/tutorials/tabular-bq-prediction/prerequisites).

You can find your Project ID in the [GCP Console](https://console.cloud.google.com) by clicking the project selector at the top of the page:

![Find your GCP Project ID](https://docs.cloud.google.com/static/vertex-ai/docs/tutorials/tabular-bq-prediction/images/find-project-id.png)

In [ ]:
# (04) Configuration
GCP_PROJECT_ID  = "replace-with-project-id"
GCP_REGION      = "asia-southeast1"
MODEL_REGION    = "us-central1"
MODEL           = "gemini-2.5-pro"
MCP_SERVER_URL  = "http://localhost:8000/mcp"

import os
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"]      = GCP_PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"]     = MODEL_REGION

In [ ]:
# (04) GCP Prerequisites
!gcloud auth login

!gcloud auth application-default login

!gcloud config set project {GCP_PROJECT_ID}

!gcloud services enable \
    run.googleapis.com \
    aiplatform.googleapis.com \
    iam.googleapis.com \
    cloudresourcemanager.googleapis.com \
    cloudbuild.googleapis.com

!gcloud auth application-default set-quota-project {GCP_PROJECT_ID}

!gcloud auth list

!gcloud config list project

In [ ]:
# (04) Import libraries for Google ADK
from google.adk.agents import Agent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

In [ ]:
# (05) Connect to MCP Server
MCP_Server = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=MCP_SERVER_URL,
    )
)

In [ ]:
# (05) Define ShopApp Agent
shopapp_agent = Agent(
    name="shopapp_agent_v1",
    model=MODEL,
    description="A ShopApp shopping assistant that helps users browse products and place orders.",
    instruction="""You are a helpful ShopApp shopping assistant. Help users search for products, 
    retrieve user profiles, and place orders. If a tool returns an error, inform the user politely. 
    If a tool is successful, present the results clearly.""",
    tools=[MCP_Server],
)

In [ ]:
# (05) Setup Runner and Session
session_service = InMemorySessionService()
runner = Runner(
    agent=shopapp_agent,
    app_name="shopapp_demo",
    session_service=session_service,
)
session = await session_service.create_session(
    app_name="shopapp_demo",
    user_id="user1",
)
print("Agent ready!")

In [ ]:
# (06) Interact with the agent
async for event in runner.run_async(
    user_id="user1",
    session_id=session.id,
    new_message=types.Content(role="user", parts=[types.Part(text="hi! I want to buy a sport watch")]),
):
    if event.is_final_response():
        print(event.content.parts[0].text)

In [ ]:
# (06) Place an order
async for event in runner.run_async(
    user_id="user1",
    session_id=session.id,
    new_message=types.Content(role="user", parts=[types.Part(text="Place an order for 1 Sport Watch for user_001")]),
):
    if event.is_final_response():
        print(event.content.parts[0].text)

---
### Section 3: Authentication

This section demonstrates how to secure your MCP server with Google Cloud Run IAM Authentication, and how authorized agents connect to it.

---

In [ ]:
# (07) Deploy MCP Server to Cloud Run with IAM auth
!gcloud run deploy shopapp-mcp-server \
    --source . \
    --region={GCP_REGION} \
    --project={GCP_PROJECT_ID} \
    --no-allow-unauthenticated \
    --port=8000 \
    --quiet

In [ ]:
# (08) Get MCP Server URL
MCP_URL = !gcloud run services describe shopapp-mcp-server \
    --region={GCP_REGION} \
    --project={GCP_PROJECT_ID} \
    --format="value(status.url)"

MCP_URL = MCP_URL[0]
print(f"MCP Server URL: {MCP_URL}")

In [ ]:
# (09) Create service account for the agent
!gcloud iam service-accounts create shopapp-agent \
    --display-name="ShopApp AI Agent" \
    --project={GCP_PROJECT_ID} \
    --quiet 2>/dev/null || true

AGENT_SERVICE_ACC = f"shopapp-agent@{GCP_PROJECT_ID}.iam.gserviceaccount.com"
print(f"Service Account: {AGENT_SERVICE_ACC}")

In [ ]:
# (09) Grant invoker role to agent service account
!gcloud run services add-iam-policy-binding shopapp-mcp-server \
    --member="serviceAccount:{AGENT_SERVICE_ACC}" \
    --role="roles/run.invoker" \
    --region={GCP_REGION} \
    --project={GCP_PROJECT_ID}

In [ ]:
# (10) Connect WITHOUT auth, expect 403 Forbidden
try:
    async with streamablehttp_client(f"{MCP_URL}/mcp") as (reader, writer, _):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            print("Connected (unexpected, should have been blocked!)")
except Exception as e:
    print(f"Connection blocked (expected): {e}")

In [ ]:
# (10) Connect WITH Google identity token, success
import google.auth
from google.auth.transport.requests import Request

credentials, _ = google.auth.default()
credentials.refresh(Request())

MCP_Server_Auth = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f"{MCP_URL}/mcp",
        headers={"Authorization": f"Bearer {credentials.id_token}"},
    )
)
print("Connected to authenticated MCP server!")

In [ ]:
# (11) Define authenticated ShopApp agent
shopapp_agent_auth = Agent(
    name="shopapp_agent_v2",
    model=MODEL,
    description="A ShopApp shopping assistant that helps users browse products and place orders.",
    instruction="""You are a helpful ShopApp shopping assistant. Help users search for products,
    retrieve user profiles, and place orders. If a tool returns an error, inform the user politely.
    If a tool is successful, present the results clearly.""",
    tools=[MCP_Server_Auth],
)

session_auth = await session_service.create_session(
    app_name="shopapp_demo",
    user_id="user2",
)
runner_auth = Runner(
    agent=shopapp_agent_auth,
    app_name="shopapp_demo",
    session_service=session_service,
)
print("Authenticated agent ready!")

In [ ]:
# (12) Interact with authenticated agent
async for event in runner_auth.run_async(
    user_id="user2",
    session_id=session_auth.id,
    new_message=types.Content(role="user", parts=[types.Part(text="Search for headphones")]),
):
    if event.is_final_response():
        print(event.content.parts[0].text)

---
### Section 4: Authorization

This section demonstrates how to enforce RBAC following the Least Privilege Principle — agents should only access what they need.

---

In [ ]:
# (13) Authorization - Agent A scoped to search_products only
MCP_Server_Agent_A = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f"{MCP_URL}/mcp",
        headers={"Authorization": f"Bearer {credentials.id_token}"},
    ),
    tool_filter=["search_products"],
)

shopapp_agent_rbac = Agent(
    name="shopapp_agent_v3",
    model=MODEL,
    description="A ShopApp shopping assistant scoped to product search only.",
    instruction="""You are a helpful ShopApp shopping assistant.
    You can only search for products. If the user asks to place an order or retrieve user profiles,
    inform them politely that you don't have that capability.
    If the tool returns an error, inform the user politely.
    If the tool is successful, present the results clearly.""",
    tools=[MCP_Server_Agent_A],
)
print("Agent A scoped to search_products only")

In [ ]:
# (14) Interact with scoped agent
session_rbac = await session_service.create_session(
    app_name="shopapp_demo",
    user_id="user3",
)
runner_rbac = Runner(
    agent=shopapp_agent_rbac,
    app_name="shopapp_demo",
    session_service=session_service,
)

async for event in runner_rbac.run_async(
    user_id="user3",
    session_id=session_rbac.id,
    new_message=types.Content(role="user", parts=[types.Part(text="Search for a backpack")]),
):
    if event.is_final_response():
        print(event.content.parts[0].text)

In [ ]:
# (14) Agent A tries to place an order
async for event in runner_rbac.run_async(
    user_id="user3",
    session_id=session_rbac.id,
    new_message=types.Content(role="user", parts=[types.Part(text="Place an order for 1 backpack for user_001")]),
):
    if event.is_final_response():
        print(event.content.parts[0].text)

---
### Section 5: Tenant Isolation - SPIFFE Identity via Vertex AI Agent Engine

This section demonstrates how to enforce strict tenant isolation between different org using the same MCP server. Data is filtered by `company_id`, Company A can never access Company B's data.

---

In [ ]:
# (15) Import Vertex AI libraries
import vertexai
from vertexai import types
from vertexai.agent_engines import AdkApp

client = vertexai.Client(
    project=GCP_PROJECT_ID,
    location=MODEL_REGION,
    http_options=dict(api_version="v1beta1"),
)
print("Vertex AI client initialised")

In [ ]:
# (15) Create staging bucket for Agent Engine (if not exists)
!gcloud storage buckets create gs://{GCP_PROJECT_ID}-staging \
    --location={MODEL_REGION} \
    --project={GCP_PROJECT_ID} 2>/dev/null || true

In [ ]:
# (16) Define the base ShopApp agent for Agent Engine
shopapp_agent = Agent(
    model=MODEL,
    name="shopapp_agent",
    instruction="""You are a helpful ShopApp shopping assistant. Help users search for products,
    retrieve user profiles, and place orders. If a tool returns an error, inform the user politely.
    If a tool is successful, present the results clearly.""",
)

In [ ]:
# (17) Deploy Company A's agent on Vertex AI Agent Engine
existing = {a.api_resource.display_name: a for a in client.agent_engines.list()}

if "shopapp-agent-company-a" in existing:
    agent_company_a = existing["shopapp-agent-company-a"]
    print(f"Reusing existing: {agent_company_a.api_resource.display_name}")
else:
    agent_company_a = client.agent_engines.create(
        agent=AdkApp(agent=shopapp_agent),
        config={
            "display_name": "shopapp-agent-company-a",
            "identity_type": types.IdentityType.AGENT_IDENTITY,
            "requirements": [
                "google-cloud-aiplatform[adk,agent_engines]",
                "pydantic",
                "cloudpickle",
            ],
            "staging_bucket": f"gs://{GCP_PROJECT_ID}-staging",
        },
    )
    print(f"Deployed: {agent_company_a.api_resource.display_name}")

print(f"SPIFFE Identity: {agent_company_a.api_resource.spec.effective_identity}")

In [ ]:
# (18) Deploy Company B's agent on Vertex AI Agent Engine
if "shopapp-agent-company-b" in existing:
    agent_company_b = existing["shopapp-agent-company-b"]
    print(f"Reusing existing: {agent_company_b.api_resource.display_name}")
else:
    agent_company_b = client.agent_engines.create(
        agent=AdkApp(agent=shopapp_agent),
        config={
            "display_name": "shopapp-agent-company-b",
            "identity_type": types.IdentityType.AGENT_IDENTITY,
            "requirements": [
                "google-cloud-aiplatform[adk,agent_engines]",
                "pydantic",
                "cloudpickle",
            ],
            "staging_bucket": f"gs://{GCP_PROJECT_ID}-staging",
        },
    )
    print(f"Deployed: {agent_company_b.api_resource.display_name}")

print(f"SPIFFE Identity: {agent_company_b.api_resource.spec.effective_identity}")

In [ ]:
# (19) Query Company A's agent
async for event in agent_company_a.async_stream_query(
    user_id="company-a-user",
    message="Search for headphones",
):
    for part in event.get("content", {}).get("parts", []):
        if "text" in part:
            print("Company A:", part["text"])

In [ ]:
# (20) Query Company B's agent
async for event in agent_company_b.async_stream_query(
    user_id="company-b-user",
    message="Search for a backpack",
):
    for part in event.get("content", {}).get("parts", []):
        if "text" in part:
            print("Company B:", part["text"])

In [ ]:
# (21) Company A's agent tries to access Company B's data
async for event in agent_company_a.async_stream_query(
    user_id="company-b-user",
    message="Search for a backpack",
):
    for part in event.get("content", {}).get("parts", []):
        if "text" in part:
            print("Company A (as B):", part["text"])

In [ ]:
# (Cleanup) Delete Agent Engine deployments
agent_company_a.delete()
agent_company_b.delete()
print("Agents deleted")